# Procena broja stanovnika iz satelitskih snimaka

ResNet-18 pretreniran na ImageNet-u, fine-tuning na Sentinel-2 isečke (6 opsega) po naselju.
Cilj je `log1p(broj_stanovnika)`. Podela na trening i validaciju ide po opštinama (GroupKFold),
da susedna naselja ne cure između skupova.

## Priprema

In [ ]:
import os, glob, zipfile
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
try:
    import timm
except ImportError:
    os.system("pip -q install timm"); import timm
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

if not os.path.exists("/content/data/naselje_table.parquet"):
    with zipfile.ZipFile("/content/data_upload.zip") as z:
        z.extractall("/content/data")

BASE, CUT = "/content/data", "/content/data/cutouts"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NB, PX = 6, 224
EP_HEAD, EP_FT, BS = 3, 50, 64
LR_HEAD, LR_FT = 1e-3, 3e-4
print("device:", DEVICE, "| cutouts:", len(os.listdir(CUT)))

## Podaci i podela

In [ ]:
# ime fajla isečka = matični broj naselja; spaja se sa tabelom labela
labele = pd.read_parquet(BASE + "/naselje_table.parquet")[
    ["naselje_maticni_broj", "opstina_maticni_broj", "pop"]]
df = pd.DataFrame({"path": glob.glob(CUT + "/*.npy")})
df["naselje_maticni_broj"] = df.path.map(lambda f: int(os.path.splitext(os.path.basename(f))[0]))
df = df.merge(labele, on="naselje_maticni_broj", how="inner")
df["y"] = np.log1p(df["pop"]).astype("float32")

splitter = GroupKFold(n_splits=min(5, df.opstina_maticni_broj.nunique()))
tr, va = next(splitter.split(df, groups=df.opstina_maticni_broj))
train_df = df.iloc[tr].reset_index(drop=True)
val_df = df.iloc[va].reset_index(drop=True)
print(f"uzoraka {len(df)} | opstina {df.opstina_maticni_broj.nunique()} | trening {len(train_df)} | validacija {len(val_df)}")

## Normalizacija i dataset

In [ ]:
# statistika po opsegu, racunata samo na trening skupu
uzorak = np.stack([np.load(p) for p in train_df.path.sample(min(400, len(train_df)), random_state=0)])
MEAN = uzorak.mean((0, 2, 3), keepdims=True).astype("float32")
STD = uzorak.std((0, 2, 3), keepdims=True).astype("float32") + 1e-6

class Naselja(Dataset):
    def __init__(self, frame, augment=False):
        self.frame = frame
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        red = self.frame.iloc[i]
        x = (np.load(red.path).astype("float32") - MEAN[0]) / STD[0]
        if self.augment:
            if np.random.rand() < 0.5: x = x[:, :, ::-1]
            if np.random.rand() < 0.5: x = x[:, ::-1, :]
            x = np.rot90(x, np.random.randint(4), axes=(1, 2))
        return torch.from_numpy(np.ascontiguousarray(x)), torch.tensor([red.y], dtype=torch.float32)

train_dl = DataLoader(Naselja(train_df, augment=True), batch_size=BS, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(Naselja(val_df), batch_size=BS, num_workers=2, pin_memory=True)

## Model

In [ ]:
# timm prosiruje prvi konvolucioni sloj sa 3 na 6 kanala (in_chans)
model = timm.create_model("resnet18", pretrained=True, in_chans=NB, num_classes=1).to(DEVICE)
loss_fn = nn.HuberLoss()

def prodji(loader, treniraj, optim=None):
    model.train(treniraj)
    ukupno, P, Y = 0.0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(treniraj):
            out = model(x)
            loss = loss_fn(out, y)
            if treniraj:
                optim.zero_grad(); loss.backward(); optim.step()
        ukupno += loss.item() * len(x)
        P.append(out.detach().cpu().numpy()); Y.append(y.cpu().numpy())
    return ukupno / len(loader.dataset), np.concatenate(P).ravel(), np.concatenate(Y).ravel()

## Trening

In [ ]:
istorija = []        # po epohi: trening gubitak, validacioni gubitak, validacioni R2
best_r2, best_state = -1e9, None

def epoha(opt):
    tl, _, _ = prodji(train_dl, True, opt)
    vl, P, Y = prodji(val_dl, False)
    r2 = r2_score(Y, P)
    istorija.append({"train_loss": tl, "val_loss": vl, "val_r2": r2})
    return tl, vl, r2

# faza 1: zamrznut backbone, uci se samo glava
for naziv, param in model.named_parameters():
    param.requires_grad = naziv.startswith("fc")
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR_HEAD)
for e in range(EP_HEAD):
    tl, vl, r2 = epoha(opt)
    print(f"[glava {e}] train {tl:.3f} val {vl:.3f} R2 {r2:.3f}")

# faza 2: odmrznut ceo model, fine-tuning sa cosine rasporedom
for param in model.parameters():
    param.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=LR_FT)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP_FT)
for e in range(EP_FT):
    tl, vl, r2 = epoha(opt); sched.step()
    if r2 > best_r2:
        best_r2 = r2
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"[fine {e}] train {tl:.3f} val {vl:.3f} R2 {r2:.3f}")

model.load_state_dict(best_state)     # vrati najbolju tezinu po validaciji
print(f"najbolji validacioni R2: {best_r2:.3f}")

## Evaluacija

In [ ]:
_, P, Y = prodji(val_dl, False)
pred, stvarno = np.expm1(P), np.expm1(Y)
print("validacija (log skala):  R2 %.3f  MAE %.3f  RMSE %.3f"
      % (r2_score(Y, P), mean_absolute_error(Y, P), mean_squared_error(Y, P) ** 0.5))
print("validacija (stanovnici): MAE %.0f  RMSE %.0f"
      % (mean_absolute_error(stvarno, pred), mean_squared_error(stvarno, pred) ** 0.5))

# agregacija predikcija na nivo opstine (Robinsonova provera)
val_df2 = val_df.assign(pred=pred, stvarno=stvarno)
po_opstini = val_df2.groupby("opstina_maticni_broj")[["pred", "stvarno"]].sum()
r2_opstina = r2_score(po_opstini.stvarno, po_opstini.pred) if len(po_opstini) > 1 else float("nan")
print("agregacija po opstini R2: %.3f" % r2_opstina)

ep = range(len(istorija))
tl = [h["train_loss"] for h in istorija]
vl = [h["val_loss"] for h in istorija]
r2 = [h["val_r2"] for h in istorija]

fig, ax = plt.subplots(2, 2, figsize=(12, 9))
ax[0, 0].plot(ep, tl, label="trening"); ax[0, 0].plot(ep, vl, label="validacija")
ax[0, 0].axvline(EP_HEAD - 0.5, ls=":", color="gray")
ax[0, 0].set_title("Huber gubitak po epohi"); ax[0, 0].set_xlabel("epoha"); ax[0, 0].legend()

ax[0, 1].plot(ep, r2); ax[0, 1].axhline(0, color="red", ls="--")
ax[0, 1].set_title("Validacioni R2 po epohi"); ax[0, 1].set_xlabel("epoha")

m = max(stvarno.max(), pred.max(), 1)
ax[1, 0].scatter(stvarno, pred, s=12, alpha=0.4); ax[1, 0].plot([1, m], [1, m], "r--")
ax[1, 0].set_xscale("log"); ax[1, 0].set_yscale("log")
ax[1, 0].set_title("Naselje: stvarno vs predvidjeno"); ax[1, 0].set_xlabel("stvarno"); ax[1, 0].set_ylabel("predvidjeno")

mm = max(po_opstini.stvarno.max(), po_opstini.pred.max(), 1)
ax[1, 1].scatter(po_opstini.stvarno, po_opstini.pred, s=35); ax[1, 1].plot([1, mm], [1, mm], "r--")
ax[1, 1].set_title("Agregacija po opstini (R2 %.2f)" % r2_opstina)
ax[1, 1].set_xlabel("stvarno"); ax[1, 1].set_ylabel("predvidjeno")
plt.tight_layout(); plt.show()

torch.save(model.state_dict(), "/content/resnet18_pop.pt")